# Modelo de RandomForest para predicción de clientes potenciales para una campaña de ventas.
## Author: Ronald Barberi (KretoN)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1pJNY49mIcSm7BHIp3MeFDCC0-ZULKxJc?usp=sharing&copy)

Nota: El Analisis exploratorio de datos que se ejecuto para entregar el DataSet listo para su modelaje, se encuentra aquí: [eda_reg_clientes_venta_bpo](https://github.com/RonaldBarberi/data_analytics/blob/main/projects/eda_reg_clientes_venta_bpo/src/eda_validator_data.ipynb)

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_curve, roc_auc_score, average_precision_score
)

In [2]:
path_main = os.getcwd()
dic_args = {
    'path_data_inp': os.path.join(path_main, '..', 'data', 'dataset_clear_model.csv'),
    'path_data_out': os.path.join(path_main, '..', 'data', 'clasificacion_clientes_nuevos.csv'),
}

In [3]:
df = pd.read_csv(dic_args['path_data_inp'])
df.info()
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   operador        500000 non-null  int64
 1   ciudad          500000 non-null  int64
 2   estrato         500000 non-null  int64
 3   rango_recarga   500000 non-null  int64
 4   equipo          500000 non-null  int64
 5   venta           500000 non-null  int64
 6   tiempo_llamada  500000 non-null  int64
 7   edad_cliente    500000 non-null  int64
 8   genero          500000 non-null  int64
 9   tipo_plan       500000 non-null  int64
 10  score_cliente   500000 non-null  int64
 11  interacciones   500000 non-null  int64
dtypes: int64(12)
memory usage: 45.8 MB


,operador,ciudad,estrato,rango_recarga,equipo,venta,tiempo_llamada,edad_cliente,genero,tipo_plan,score_cliente,interacciones
0,1,1,4,3,0,1,4,2,0,0,1,17
1,3,4,3,2,3,0,4,0,1,1,2,20
2,0,6,3,4,4,0,4,3,1,0,3,11
3,2,6,2,4,6,0,4,3,1,1,2,19
4,1,1,6,4,6,0,2,2,1,0,2,25


In [4]:
df_valid = df.groupby('venta')['venta'].count()
df_valid

venta
0    300366
1    199634
Name: venta, dtype: int64

### Variable predictoria y objetivo

In [5]:
X = df.drop(columns=['venta'])
Y = df['venta']

### Dividir datos de entrenamiento y prueba.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

### Undersampling para balancear la información.

In [7]:
rus = RandomUnderSampler(
    sampling_strategy={0: 160_000},
    random_state=42
)
X_res, y_res = rus.fit_resample(X_train, y_train)

print("Distribución tras undersampling:")
print(y_res.value_counts())

Distribución tras undersampling:
venta
0    160000
1    159707
Name: count, dtype: int64


In [8]:
model_rf = RandomForestClassifier(
    n_estimators=537,
    max_depth=20,
    max_features='log2',
    min_samples_leaf=5,
    n_jobs=-1,
    class_weight={0: 1, 1: 1.02},
    random_state=42,
)

In [9]:
model_rf.fit(X_res, y_res)

,n_estimators,537
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
y_pred = model_rf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.49293
[[27778 32295]
 [18412 21515]]
              precision    recall  f1-score   support

           0       0.60      0.46      0.52     60073
           1       0.40      0.54      0.46     39927

    accuracy                           0.49    100000
   macro avg       0.50      0.50      0.49    100000
weighted avg       0.52      0.49      0.50    100000



In [11]:
importances = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': model_rf.feature_importances_
}).sort_values(by='Importancia', ascending=False)
print(importances.head(10))

          Variable  Importancia
10   interacciones     0.229851
1           ciudad     0.097791
2          estrato     0.097181
4           equipo     0.094986
0         operador     0.089796
5   tiempo_llamada     0.089225
3    rango_recarga     0.082781
6     edad_cliente     0.073628
9    score_cliente     0.070434
7           genero     0.044998
